In [ ]:
!pip -q install pandas openai-whisper
!apt-get -qq update
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.5 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import json, random, datetime, uuid, os

INTENTS = ["benefits", "prior_auth", "claim_status", "billing", "provider_network"]
PAYERS = ["Aetna", "BCBS", "UnitedHealthcare", "Cigna", "Humana"]
PROVIDERS = ["MGH", "Brigham", "Beth Israel", "Tufts Medical", "Lahey"]
CPT_CODES = ["99214", "90791", "90834", "90837", "93000", "72148"]
ISSUES = {
    "benefits": ["check coverage for outpatient mental health", "verify deductible and copay", "confirm in-network benefits"],
    "prior_auth": ["prior authorization status for therapy sessions", "documents needed for authorization", "urgent authorization request"],
    "claim_status": ["claim denied and needs appeal steps", "claim pending for more than 30 days", "request claim payment status"],
    "billing": ["billing code mismatch and patient balance", "invoice clarification", "request itemized bill"],
    "provider_network": ["confirm provider is in-network", "check facility coverage", "update provider directory"]
}

def masked_member_id():
    return f"***{random.randint(1000,9999)}"

def random_date(start_days_ago=120):
    d = datetime.date.today() - datetime.timedelta(days=random.randint(1, start_days_ago))
    return d.isoformat()

def random_dob():
    year = random.randint(1970, 2005)
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    return f"{year:04d}-{month:02d}-{day:02d}"

def make_transcript(intent, payer, provider, member_id, dob, cpt, dos, include_disclosure=True, include_id_verify=True):
    disclosure = "This call may be recorded for quality purposes. " if include_disclosure else ""
    id_part = f"My name is Alex. Date of birth is {dob}. Member ID is {member_id}. " if include_id_verify else ""
    issue = random.choice(ISSUES[intent])
    return (
        f"{disclosure}Hi, I'm calling about {issue}. {id_part}"
        f"The payer is {payer}. Provider is {provider}. CPT code {cpt}. Date of service {dos}. "
        f"Can you confirm the details and tell me next steps?"
    )

def generate_calls(n=100):
    records = []
    for _ in range(n):
        call_id = str(uuid.uuid4())[:8]
        intent = random.choice(INTENTS)
        payer = random.choice(PAYERS)
        provider = random.choice(PROVIDERS)
        member_id = masked_member_id()
        dob = random_dob()
        cpt = random.choice(CPT_CODES)
        dos = random_date()

        include_disclosure = random.random() > 0.15
        include_id_verify = random.random() > 0.10

        transcript = make_transcript(intent, payer, provider, member_id, dob, cpt, dos,
                                     include_disclosure=include_disclosure,
                                     include_id_verify=include_id_verify)

        qa_labels = {
            "qa_recording_disclosed": int(include_disclosure),
            "qa_identity_verified": int(include_id_verify),
            "qa_next_step_requested": 1
        }

        records.append({
            "call_id": call_id,
            "intent_true": intent,
            "transcript": transcript,
            "qa_labels": qa_labels
        })
    return records

calls = generate_calls(100)
print("Sample record:\n", json.dumps(calls[0], indent=2))

Sample record:
 {
  "call_id": "f8cc19a6",
  "intent_true": "prior_auth",
  "transcript": "This call may be recorded for quality purposes. Hi, I'm calling about prior authorization status for therapy sessions. My name is Alex. Date of birth is 1991-08-01. Member ID is ***4922. The payer is Aetna. Provider is Beth Israel. CPT code 93000. Date of service 2025-12-16. Can you confirm the details and tell me next steps?",
  "qa_labels": {
    "qa_recording_disclosed": 1,
    "qa_identity_verified": 1,
    "qa_next_step_requested": 1
  }
}


In [ ]:
!pip -q uninstall -y click
!pip -q install "click>=8.2.1" "typer==0.24.1"
!python -c "import click, typer; print('click:', click.__version__, 'typer:', typer.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
<string>:1: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Click 9.1. Use feature detection or 'importlib.metadata.version("click")' instead.
click: 8.3.1 typer: 0.24.1


In [ ]:
!pip -q install gTTS
from gtts import gTTS

text = "This call may be recorded for quality purposes. My name is Alex. Date of birth is 1998-04-10. Member ID is star star star one two three four. The payer is Aetna. Provider is MGH. CPT code 90837. Date of service 2026-02-10. I am calling to check prior authorization status. What are the next steps?"
gTTS(text).save("demo_call.mp3")
print("Saved demo_call.mp3")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Saved demo_call.mp3


In [ ]:
import whisper

model = whisper.load_model("base")
result = model.transcribe("demo_call.mp3")
transcript_from_audio = result["text"].strip()

print(transcript_from_audio)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


This call may be recorded for quality purposes. My name is Alex. Date of birth is 10 April 1998. Member ID is star star star 1 2 3 4. The pair is Etna. Provider is MGH. CPT Code 90837. Date of service 10 February 2026. I am calling to check prior authorization status. What are the next steps?


In [ ]:
import re

# ---------- Intent + Field Extraction ----------
INTENT_KEYWORDS = {
    "benefits": ["coverage", "deductible", "copay", "benefits", "in-network"],
    "prior_auth": ["prior authorization", "authorization", "auth"],
    "claim_status": ["claim", "denied", "appeal", "payment status", "pending"],
    "billing": ["invoice", "bill", "itemized", "balance", "billing"],
    "provider_network": ["provider", "in-network", "directory", "facility coverage"]
}

def predict_intent(transcript: str) -> str:
    t = transcript.lower()
    scores = {k: 0 for k in INTENT_KEYWORDS}
    for intent, kws in INTENT_KEYWORDS.items():
        for kw in kws:
            if kw in t:
                scores[intent] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unknown"

def extract_fields(transcript: str) -> dict:
    # More flexible patterns (works even if Whisper changes punctuation)
    payer = re.search(r"payer\s+is\s+([A-Za-z]+)", transcript, re.IGNORECASE)
    provider = re.search(r"provider\s+is\s+([A-Za-z ]+?)(?:\.|,|cpt|date|$)", transcript, re.IGNORECASE)
    dob = re.search(r"date\s+of\s+birth\s+is\s+(.+?)(?:\.|,|member|payer|provider|$)", transcript, re.IGNORECASE)
    member = re.search(r"member\s+id\s+is\s+(.+?)(?:\.|,|payer|provider|cpt|date|$)", transcript, re.IGNORECASE)
    cpt = re.search(r"cpt\s*code\s*(\d+)", transcript, re.IGNORECASE)
    dos = re.search(r"date\s+of\s+service\s+(.+?)(?:\.|,|$)", transcript, re.IGNORECASE)

    return {
        "intent_pred": predict_intent(transcript),
        "payer": payer.group(1).strip() if payer else None,
        "provider": provider.group(1).strip() if provider else None,
        "dob": dob.group(1).strip() if dob else None,
        "member_id_masked": member.group(1).strip() if member else None,
        "cpt_code": cpt.group(1).strip() if cpt else None,
        "date_of_service": dos.group(1).strip() if dos else None
    }

# ---------- Ticket Generator ----------
def make_ticket(call_id: str, fields: dict) -> dict:
    intent = fields.get("intent_pred", "unknown")
    payer = fields.get("payer") or "UNKNOWN"
    provider = fields.get("provider") or "UNKNOWN"
    cpt = fields.get("cpt_code") or "UNKNOWN"
    dos = fields.get("date_of_service") or "UNKNOWN"

    summary = f"{intent.upper()} request for payer {payer}. Provider: {provider}, CPT: {cpt}, DOS: {dos}."
    next_action = {
        "benefits": "Verify coverage details and provide deductible/copay info.",
        "prior_auth": "Check prior auth status and required documentation.",
        "claim_status": "Check claim status and provide resolution/appeal steps.",
        "billing": "Review billing details and provide itemized explanation.",
        "provider_network": "Confirm network status and update directory details if needed.",
        "unknown": "Review request details and follow up."
    }.get(intent, "Review request details and follow up.")

    return {
        "call_id": call_id,
        "ticket_title": f"{intent} - {payer}",
        "summary": summary,
        "recommended_next_action": next_action,
        "extracted_fields": fields
    }

# ---------- QA Monitor ----------
REQUIRED_FIELDS = ["payer", "provider", "cpt_code", "date_of_service"]

def qa_report(transcript: str, fields: dict) -> dict:
    t = transcript.lower()
    recording_disclosed = int("recorded for quality" in t)
    identity_verified = int(("date of birth" in t) and ("member id" in t))
    next_step_requested = int(("next step" in t) or ("next steps" in t))

    missing_fields = [f for f in REQUIRED_FIELDS if not fields.get(f)]
    qa_score = recording_disclosed + identity_verified + next_step_requested

    return {
        "qa_recording_disclosed": recording_disclosed,
        "qa_identity_verified": identity_verified,
        "qa_next_step_requested": next_step_requested,
        "missing_fields": missing_fields,
        "qa_score": qa_score
    }

print("✅ Functions loaded: extract_fields, make_ticket, qa_report")

✅ Functions loaded: extract_fields, make_ticket, qa_report


In [ ]:
# Re-generate the synthetic dataset list
calls = generate_calls(100)   # or any number like 50/200
print("calls loaded:", len(calls))
print(calls[0].keys())

calls loaded: 100
dict_keys(['call_id', 'intent_true', 'transcript', 'qa_labels'])


In [ ]:
import json, pandas as pd, os

os.makedirs("out", exist_ok=True)

rows = []
with open("out/extracted_calls.jsonl", "w", encoding="utf-8") as f:
    for r in calls:  # your synthetic dataset list
        transcript = r["transcript"]
        fields = extract_fields(transcript)  # or extract_fields_improved(transcript)
        ticket = make_ticket(r["call_id"], fields)
        qa = qa_report(transcript, fields)

        f.write(json.dumps({"call_id": r["call_id"], "intent_true": r["intent_true"], "ticket": ticket, "qa": qa}) + "\n")

        rows.append({
            "call_id": r["call_id"],
            "intent_true": r["intent_true"],
            "intent_pred": fields.get("intent_pred"),
            "qa_score": qa.get("qa_score"),
            "missing_fields": "|".join(qa.get("missing_fields", [])),
            "recording_disclosed": qa.get("qa_recording_disclosed"),
            "identity_verified": qa.get("qa_identity_verified"),
            "next_step_requested": qa.get("qa_next_step_requested")
        })

df = pd.DataFrame(rows)
df.to_csv("out/qa_report.csv", index=False)

print("Saved: out/extracted_calls.jsonl and out/qa_report.csv")
df.head()

Saved: out/extracted_calls.jsonl and out/qa_report.csv


,call_id,intent_true,intent_pred,qa_score,missing_fields,recording_disclosed,identity_verified,next_step_requested
0,169f2006,benefits,benefits,3,,1,1,1
1,3adc96ac,provider_network,provider_network,3,,1,1,1
2,548a228b,provider_network,provider_network,3,,1,1,1
3,699cb005,billing,billing,3,,1,1,1
4,d9deaeb1,provider_network,provider_network,2,,0,1,1


In [ ]:
from google.colab import files
files.download("out/extracted_calls.jsonl")
files.download("out/qa_report.csv")
files.download("demo_call.mp3")   # optional: audio demo file

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Intent accuracy
intent_acc = (df["intent_true"] == df["intent_pred"]).mean()
print("Intent prediction accuracy:", round(intent_acc*100, 2), "%")

# QA score distribution
print(df["qa_score"].value_counts().sort_index())

Intent prediction accuracy: 100.0 %
qa_score
1     3
2    22
3    75
Name: count, dtype: int64


In [ ]:
import json

# take one record from your processed output
sample = calls[0]
transcript = sample["transcript"]
fields = extract_fields(transcript)
ticket = make_ticket(sample["call_id"], fields)
qa = qa_report(transcript, fields)

demo = {"call_id": sample["call_id"], "transcript": transcript, "ticket": ticket, "qa": qa}
with open("out/demo_output.json", "w") as f:
    json.dump(demo, f, indent=2)

from google.colab import files
files.download("out/demo_output.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>